In [1]:
# %% [markdown]
# # Ray Tune 最终 Epoch 结算分析函数 (含 Temp 温度/EFIN/S4 全兼容版)
# 逻辑：传入 base_path，动态抓取 head_hidden_dims、alpha 以及各版本特异性 temp 温度指标。

# %%
import os
import json
import pandas as pd

# %%
def analyze_ray_results(base_path):
    """
    遍历指定的 Ray 结果路径，按模型实际超参分类对比最后 Epoch 的结算指标。
    🌟 完美补齐：自动提取 ours_s6_temp 等各种冲突温度指标，拒绝张冠李戴。
    """
    print(f"===========================================================")
    print(f" 开始解析路径: {base_path}")
    print(f"===========================================================")
    
    all_trials_data = []

    # 遍历目录
    for root, dirs, files in os.walk(base_path):
        if "params.json" in files and "result.json" in files:
            params_path = os.path.join(root, "params.json")
            result_path = os.path.join(root, "result.json")
            
            # 1. 解析参数
            try:
                with open(params_path, 'r', encoding='utf-8') as f:
                    params = json.load(f)
            except Exception:
                continue
                
            # 2. 定位最后一轮的结算指标
            last_epoch_data = None
            try:
                with open(result_path, 'r', encoding='utf-8') as f:
                    lines = [line.strip() for line in f if line.strip()]
                    if lines:
                        # 直接获取最后一行（最后 Epoch）
                        last_epoch_data = json.loads(lines[-1])
            except Exception:
                continue
                
            if last_epoch_data is not None:
                # 转换 list 为 str 方便后续做分类聚合
                head_dims = params.get("head_hidden_dims")
                head_dims_str = str(head_dims) if head_dims is not None else "None"
                
                # 提取最终结算数据 (引入全部新模型变量与核心 temp 变量兜底)
                trial_info = {
                    "trial_id": last_epoch_data.get("trial_id", os.getpid()),
                    "model": params.get("model", "None"),
                    "efin_embed_dim": params.get("efin_embed_dim", "None"), 
                    "head_hidden_dims": head_dims_str,                     
                    "alpha_wool": params.get("conflict_alpha_wool", "None"),
                    "alpha_gold": params.get("conflict_alpha_gold", "None"),
                    "alpha_walkin": params.get("conflict_alpha_walkin", "None"),
                    # 🌟 绝杀锁定：提取大盘消融矩阵中的各类温度参数 (支持 s6_temp 等)
                    "temp": params.get("ours_s6_temp", params.get("truncation_temp", params.get("align_temp", "None"))),
                    "weight_decay": params.get("weight_decay", "None"),
                    "final_epoch": last_epoch_data.get("epoch"),
                    "final_local_best_test_Y_AUUC": last_epoch_data.get("local_best_test_Target_Y_AUUC"),
                    "final_local_best_test_C_AUUC": last_epoch_data.get("local_best_test_Target_C_AUUC"),
                    "final_loss": last_epoch_data.get("loss")
                }
                all_trials_data.append(trial_info)

    # 3. 统计并打印结果
    df = pd.DataFrame(all_trials_data)

    if df.empty:
        print("❌ 没有找到任何有效的实验数据，请检查传入的路径是否正确。\n")
        return None
        
    # 🌟 关键补齐：将核心分类参数加入 temp
    core_params = ["model","head_hidden_dims", "alpha_wool", "alpha_gold", "alpha_walkin", "temp", "weight_decay"]
    for col in core_params:
        if col in df.columns:
            df[col] = df[col].fillna("None").astype(str)
        else:
            df[col] = "None"

    print(f"📊 成功加载 {len(df)} 个实验的最终结算数据。\n")
    
    # -------------------------------------------------------------
    # 维度 1：按参数组合展现各自最后 Epoch 的最高表现
    # -------------------------------------------------------------
    print("="*75)
    print(" 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC ")
    print("="*75)
    
    idx = df.groupby(core_params)["final_local_best_test_Y_AUUC"].idxmax()
    df_best_per_group = df.loc[idx].sort_values(by="final_local_best_test_Y_AUUC", ascending=False)
    
    for _, row in df_best_per_group.iterrows():
        print(f"\n[配置组合] Model: {row['model']}  | Head: {row['head_hidden_dims']}")
        print(f"            alpha_w/g/wk: {row['alpha_wool']}/{row['alpha_gold']}/{row['alpha_walkin']} | Temp: {row['temp']} | wd: {row['weight_decay']}")
        print(f"  -> 最终结算 Epoch: {row['final_epoch']} (Loss: {row['final_loss']:.4f})")
        print(f"  -> 最终 local_best_test_Y_AUUC: {row['final_local_best_test_Y_AUUC']:.6f}")
        print(f"  -> 最终 local_best_test_C_AUUC: {row['final_local_best_test_C_AUUC']:.6f}")
        print(f"  -> Trial ID: {row['trial_id']}")

    # -------------------------------------------------------------
    # 维度 2：大宽表横向对比
    # -------------------------------------------------------------
    print("\n" + "="*75)
    print(" 📋 全局最终战报总览表（按最终 local_best_test_Y_AUUC 从高到低排序） ")
    print("="*75)
    
    report_cols = core_params + ["final_epoch", "final_local_best_test_Y_AUUC", "final_local_best_test_C_AUUC"]
    print(df.sort_values(by="final_local_best_test_Y_AUUC", ascending=False)[report_cols].to_string(index=False))
    print("\n")
    
    return df

In [2]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_all_same_alpha_head_0710/exp_y_pure_v10_all_same_alpha_head_0710_res_2_layer_head_search"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_all_same_alpha_head_0710/exp_y_pure_v10_all_same_alpha_head_0710_res_2_layer_head_search
📊 成功加载 30 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET_Baseline_PureV10  | Head: [32]
            alpha_w/g/wk: 1.0/1.0/1.0 | Temp: None | wd: 0.0001
  -> 最终结算 Epoch: 30 (Loss: 0.0122)
  -> 最终 local_best_test_Y_AUUC: 0.922116
  -> 最终 local_best_test_C_AUUC: 0.823849
  -> Trial ID: 141ec_00015

[配置组合] Model: TARNET_Baseline_PureV10  | Head: [16]
            alpha_w/g/wk: 5.0/5.0/5.0 | Temp: None | wd: 1e-05
  -> 最终结算 Epoch: 16 (Loss: 0.0130)
  -> 最终 local_best_test_Y_AUUC: 0.913983
  -> 最终 local_best_test_C_AUUC: 0.827496
  -> Trial ID: 141ec_00013

[配置组合] Model: TARNET_Baseline_PureV10  | Head: [64, 32]
            alpha_w/g/wk: 0.5/0.5/0.5 | Temp: None | wd: 0.0001
  -> 最终结算 Epoch: 19 (Loss: 0.0131)
  -> 最终 local_best_test_Y_AUUC: 0.912334
  -> 最终 local_best_test_

In [3]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_mix_all_same_alpha/exp_v10_mix_all/exp_v10_mix_all"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET_Baseline_PureV10/y_pure_v10_mix_all_same_alpha/exp_v10_mix_all/exp_v10_mix_all
📊 成功加载 84 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET_Baseline_PureV10  | Head: None
            alpha_w/g/wk: 0.5/5.0/1.0 | Temp: None | wd: 1e-05
  -> 最终结算 Epoch: 14 (Loss: 0.0456)
  -> 最终 local_best_test_Y_AUUC: 0.907201
  -> 最终 local_best_test_C_AUUC: 0.824569
  -> Trial ID: 69a98_00038

[配置组合] Model: TARNET_Baseline_PureV10  | Head: None
            alpha_w/g/wk: 0.5/10.0/0.5 | Temp: None | wd: 1e-05
  -> 最终结算 Epoch: 14 (Loss: 0.0628)
  -> 最终 local_best_test_Y_AUUC: 0.907155
  -> 最终 local_best_test_C_AUUC: 0.818229
  -> Trial ID: 69a98_00034

[配置组合] Model: TARNET_Baseline_PureV10  | Head: None
            alpha_w/g/wk: 0.5/0.5/0.1 | Temp: None | wd: 1e-05
  -> 最终结算 Epoch: 20 (Loss: 0.0166)
  -> 最终 local_best_test_Y_AUUC: 0.906704
  -> 最终 local_best_test_C_AUUC: 0.801980
  -> Trial ID: 69a98_00026

In [4]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s6_conflict_0710_alpha_search_temp1_20/y_ours_s6_conflict_0710_alpha_search_temp1_20"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s6_conflict_0710_alpha_search_temp1_20/y_ours_s6_conflict_0710_alpha_search_temp1_20


📊 成功加载 200 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET  | Head: [32]
            alpha_w/g/wk: 10.0/10.0/10.0 | Temp: 1 | wd: 0.01
  -> 最终结算 Epoch: 11 (Loss: 0.0390)
  -> 最终 local_best_test_Y_AUUC: 0.913785
  -> 最终 local_best_test_C_AUUC: 0.830434
  -> Trial ID: 82204_00093

[配置组合] Model: TARNET  | Head: [32]
            alpha_w/g/wk: 0.1/0.1/0.1 | Temp: 20 | wd: 1e-05
  -> 最终结算 Epoch: 21 (Loss: 0.0132)
  -> 最终 local_best_test_Y_AUUC: 0.912299
  -> 最终 local_best_test_C_AUUC: 0.803177
  -> Trial ID: 82204_00019

[配置组合] Model: TARNET  | Head: [32]
            alpha_w/g/wk: 0.5/0.5/0.5 | Temp: 20 | wd: 1e-05
  -> 最终结算 Epoch: 19 (Loss: 0.0162)
  -> 最终 local_best_test_Y_AUUC: 0.910768
  -> 最终 local_best_test_C_AUUC: 0.829934
  -> Trial ID: 82204_00016

[配置组合] Model: TARNET  | Head: [32]
            alpha_w/g/wk: 10.0/10.0/10.0 | Temp: 20 | wd: 0.01
  -> 最终结算 Epoch: 15 (Loss: 0.0772)
  -> 最终 local_best_test_Y_AUUC: 0.909989
  -> 最终 local_best_test_C

In [5]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s4_conflict_0710/y_ours_s4_conflict_0710n/y_ours_s4_conflict_0710n"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_ours_s4_conflict_0710/y_ours_s4_conflict_0710n/y_ours_s4_conflict_0710n
📊 成功加载 9 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 0.5/0.5/0.5 | Temp: None | wd: 1e-05
  -> 最终结算 Epoch: 18 (Loss: 0.0160)
  -> 最终 local_best_test_Y_AUUC: 0.905981
  -> 最终 local_best_test_C_AUUC: 0.818934
  -> Trial ID: d2962_00002

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 1.0/1.0/1.0 | Temp: None | wd: 0.0001
  -> 最终结算 Epoch: 15 (Loss: 0.0207)
  -> 最终 local_best_test_Y_AUUC: 0.905654
  -> 最终 local_best_test_C_AUUC: 0.821452
  -> Trial ID: d2962_00003

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 5.0/5.0/5.0 | Temp: None | wd: 0.0001
  -> 最终结算 Epoch: 15 (Loss: 0.0374)
  -> 最终 local_best_test_Y_AUUC: 0.902570
  -> 最终 local_best_test_C_AUUC: 0.821288
  -> Trial ID: d2962_00004

[配置组合] Model: TARNET  | Head: None
            alpha_w

In [6]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_v8_s4_v10_all_mix_original_code_search_head_wd_alpha_same"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_v8_s4_v10_all_mix_original_code_search_head_wd_alpha_same
📊 成功加载 19 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 0.0/0.0/0.0 | Temp: None | wd: 1e-05
  -> 最终结算 Epoch: 16 (Loss: 0.0132)
  -> 最终 local_best_test_Y_AUUC: 0.908725
  -> 最终 local_best_test_C_AUUC: 0.830358
  -> Trial ID: ca11b_00011

[配置组合] Model: TARNET  | Head: [32]
            alpha_w/g/wk: 10.0/10.0/10.0 | Temp: None | wd: 1e-05
  -> 最终结算 Epoch: 15 (Loss: 0.0345)
  -> 最终 local_best_test_Y_AUUC: 0.908300
  -> 最终 local_best_test_C_AUUC: 0.821262
  -> Trial ID: ca11b_00003

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 10.0/10.0/10.0 | Temp: None | wd: 0.0001
  -> 最终结算 Epoch: 3 (Loss: 0.0804)
  -> 最终 local_best_test_Y_AUUC: 0.905088
  -> 最终 local_best_test_C_AUUC: 0.826618
  -> Trial ID: aa079_00009

[配置组合] Model: TARNET  | Head: [32]
            alpha_w/g/wk: 1.

In [7]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_v8_s6_temp_10_20_v10_all_mix_original_code_search_head_wd_alpha_same"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_v8_s6_temp_10_20_v10_all_mix_original_code_search_head_wd_alpha_same
📊 成功加载 42 个实验的最终结算数据。

 🎯 按照核心超参数分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 0.5/0.5/0.5 | Temp: 20 | wd: 1e-05
  -> 最终结算 Epoch: 11 (Loss: 0.0179)
  -> 最终 local_best_test_Y_AUUC: 0.908096
  -> 最终 local_best_test_C_AUUC: 0.831720
  -> Trial ID: ed5cb_00015

[配置组合] Model: TARNET  | Head: [32]
            alpha_w/g/wk: 0.5/0.5/0.5 | Temp: 10 | wd: 1e-05
  -> 最终结算 Epoch: 17 (Loss: 0.0168)
  -> 最终 local_best_test_Y_AUUC: 0.906629
  -> 最终 local_best_test_C_AUUC: 0.827200
  -> Trial ID: ed5cb_00002

[配置组合] Model: TARNET  | Head: None
            alpha_w/g/wk: 10.0/10.0/10.0 | Temp: 10 | wd: 1e-05
  -> 最终结算 Epoch: 14 (Loss: 0.0547)
  -> 最终 local_best_test_Y_AUUC: 0.905936
  -> 最终 local_best_test_C_AUUC: 0.820726
  -> Trial ID: ed5cb_00018

[配置组合] Model: TARNET  | Head: [32]
            alpha_w/g/wk: 